# Deep Retention — Limpieza y preprocesamiento de datos

Este notebook concentra la preparación reproducible de los datos. Carga el CSV original, limpia una copia, conserva una versión legible sin escalado, reproduce las particiones 70/15/15 y ajusta el `ColumnTransformer` únicamente con entrenamiento.

Al finalizar exporta el CSV limpio, las tres particiones transformadas, el preprocesador ajustado y metadatos de trazabilidad. No modifica el archivo original y no entrena modelos.

## 1. Carga y limpieza en memoria

La limpieza conserva las 10.000 filas y las 21 columnas: elimina espacios exteriores en textos y representa como faltante el apellido completo de los 92 registros que contienen `?`. No reconstruye apellidos, elimina clientes ni modifica `CustomerId`.

In [1]:
import hashlib
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
SCHEMA_VERSION = "1.0"
DATA_FILENAME = "Deep_Retention_Dataset.csv"
DATA_CANDIDATES = [
    Path("data") / DATA_FILENAME,
    Path("..") / "data" / DATA_FILENAME,
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"No se encontró {DATA_FILENAME} desde la raíz ni desde notebooks/.")

PROJECT_ROOT = DATA_PATH.parent.parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_DATA_PATH = PROCESSED_DIR / "clientes_limpios.csv"
TRAIN_PATH = PROCESSED_DIR / "train.npz"
VALIDATION_PATH = PROCESSED_DIR / "validation.npz"
TEST_PATH = PROCESSED_DIR / "test.npz"
METADATA_PATH = PROCESSED_DIR / "metadata.json"
PREPROCESSOR_PATH = MODELS_DIR / "preprocesador.joblib"

FEATURES = [
    "CreditScore", "Geography", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary",
]
TARGET = "Exited"
NUMERIC_FEATURES = [
    "CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary",
]
CATEGORICAL_FEATURES = ["Geography"]
BINARY_FEATURES = ["HasCrCard", "IsActiveMember"]
EXCLUDED_FEATURES = ["CustomerId", "Surname", "Gender", TARGET]

print(f"Archivo original: {DATA_PATH.as_posix()}")
print(f"Scikit-learn: {sklearn.__version__}")

Archivo original: ../data/Deep_Retention_Dataset.csv
Scikit-learn: 1.9.1


In [2]:
df_original = pd.read_csv(DATA_PATH)
df_clean = df_original.copy()

text_columns = df_clean.select_dtypes(include="object").columns
for column in text_columns:
    df_clean[column] = df_clean[column].str.strip()

corrupted_surnames = df_clean["Surname"].str.contains("?", regex=False, na=False)
corrupted_surname_count = int(corrupted_surnames.sum())
df_clean.loc[corrupted_surnames, "Surname"] = pd.NA

numeric_types_valid = all(
    pd.api.types.is_numeric_dtype(df_clean[column])
    for column in df_clean.select_dtypes(exclude="object").columns
)

cleaning_summary = pd.DataFrame({
    "control": [
        "Filas originales", "Filas después de limpiar",
        "Columnas originales", "Columnas después de limpiar",
        "Apellidos con '?' detectados", "Apellidos con '?' restantes",
        "Valores faltantes originales", "Valores faltantes después de limpiar",
        "Faltantes en Surname", "CustomerId modificados",
        "Tipos numéricos válidos",
    ],
    "resultado": [
        len(df_original), len(df_clean), df_original.shape[1], df_clean.shape[1],
        corrupted_surname_count,
        int(df_clean["Surname"].str.contains("?", regex=False, na=False).sum()),
        int(df_original.isna().sum().sum()), int(df_clean.isna().sum().sum()),
        int(df_clean["Surname"].isna().sum()),
        int((df_clean["CustomerId"] != df_original["CustomerId"]).sum()),
        numeric_types_valid,
    ],
})
display(cleaning_summary)

,control,resultado
0,Filas originales,10000
1,Filas después de limpiar,10000
2,Columnas originales,21
3,Columnas después de limpiar,21
4,Apellidos con '?' detectados,92
5,Apellidos con '?' restantes,0
6,Valores faltantes originales,0
7,Valores faltantes después de limpiar,92
8,Faltantes en Surname,92
9,CustomerId modificados,0


**Resultado.** La limpieza se aplica solo a `df_clean`. Los 92 apellidos problemáticos pasan a `pd.NA`; las nueve predictoras permanecen completas y los 10.000 identificadores se conservan.

## 2. Exportación del CSV limpio

`clientes_limpios.csv` conserva las 21 columnas y sus valores numéricos originales. No contiene escalado ni codificación one-hot. Los apellidos desconocidos se escriben como campos vacíos y se recuperan como valores faltantes al volver a cargar el CSV.

In [3]:
df_clean.to_csv(CLEAN_DATA_PATH, index=False, na_rep="")
clean_data_reloaded = pd.read_csv(CLEAN_DATA_PATH)

clean_csv_checks = pd.DataFrame({
    "comprobación": [
        "10.000 registros", "21 columnas", "92 apellidos faltantes",
        "CustomerId conservados", "Valores numéricos conservados",
    ],
    "resultado": [
        len(clean_data_reloaded) == 10_000,
        clean_data_reloaded.shape[1] == 21,
        int(clean_data_reloaded["Surname"].isna().sum()) == 92,
        clean_data_reloaded["CustomerId"].equals(df_original["CustomerId"]),
        bool(np.allclose(
            clean_data_reloaded[NUMERIC_FEATURES].to_numpy(),
            df_original[NUMERIC_FEATURES].to_numpy(),
        )),
    ],
})
display(clean_csv_checks)
print(f"CSV limpio: {CLEAN_DATA_PATH.as_posix()}")

,comprobación,resultado
0,10.000 registros,True
1,21 columnas,True
2,92 apellidos faltantes,True
3,CustomerId conservados,True
4,Valores numéricos conservados,True


CSV limpio: ../data/processed/clientes_limpios.csv


## 3. Selección de variables

La matriz `X` contiene exclusivamente las nueve predictoras aprobadas. `CustomerId`, `Surname`, `Gender`, `Exited` y todas las columnas con sufijo `_Sim` permanecen en el CSV limpio, pero no ingresan al modelo.

In [4]:
X = df_clean[FEATURES].copy()
y = df_clean[TARGET].copy()

financial_columns = [column for column in df_clean.columns if column.endswith("_Sim")]
excluded_columns_in_X = sorted(
    set(X.columns) & set(EXCLUDED_FEATURES + financial_columns)
)

if X.columns.tolist() != FEATURES or X.shape[1] != 9:
    raise ValueError("X no contiene exactamente las nueve predictoras aprobadas.")
if excluded_columns_in_X:
    raise ValueError(f"X contiene columnas excluidas: {excluded_columns_in_X}")
if X.isna().any().any() or y.isna().any():
    raise ValueError("Las predictoras o el objetivo contienen valores faltantes.")

selection_summary = pd.DataFrame({
    "objeto": ["X", "y"],
    "filas": [X.shape[0], y.shape[0]],
    "columnas": [X.shape[1], 1],
    "faltantes": [int(X.isna().sum().sum()), int(y.isna().sum())],
})
display(selection_summary)
print("Variables de X:", X.columns.tolist())
print("Columnas excluidas presentes en X:", excluded_columns_in_X)

,objeto,filas,columnas,faltantes
0,X,10000,9,0
1,y,10000,1,0


Variables de X: ['CreditScore', 'Geography', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
Columnas excluidas presentes en X: []


## 4. División de entrenamiento, validación y prueba

Se reproduce el procedimiento aprobado: primero se reserva el 30 % y luego se divide esa parte por la mitad. Ambas operaciones usan `stratify` y `random_state=42`, por lo que se conservan exactamente las particiones anteriores.

In [5]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

split_summary = pd.DataFrame({
    "conjunto": ["Entrenamiento", "Validación", "Prueba"],
    "registros": [len(X_train), len(X_val), len(X_test)],
    "porcentaje_dataset": [
        len(X_train) / len(X) * 100,
        len(X_val) / len(X) * 100,
        len(X_test) / len(X) * 100,
    ],
    "abandonos": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
    "porcentaje_abandono": [
        y_train.mean() * 100,
        y_val.mean() * 100,
        y_test.mean() * 100,
    ],
})
display(split_summary)

train_indices = set(X_train.index)
validation_indices = set(X_val.index)
test_indices = set(X_test.index)
split_checks = pd.DataFrame({
    "comprobación": [
        "Entrenamiento ∩ validación", "Entrenamiento ∩ prueba",
        "Validación ∩ prueba", "Suma de tamaños",
    ],
    "resultado": [
        len(train_indices & validation_indices), len(train_indices & test_indices),
        len(validation_indices & test_indices), len(X_train) + len(X_val) + len(X_test),
    ],
    "esperado": [0, 0, 0, len(X)],
})
display(split_checks)

,conjunto,registros,porcentaje_dataset,abandonos,porcentaje_abandono
0,Entrenamiento,7000,70.0,1426,20.371429
1,Validación,1500,15.0,306,20.400000
2,Prueba,1500,15.0,305,20.333333


,comprobación,resultado,esperado
0,Entrenamiento ∩ validación,0,0
1,Entrenamiento ∩ prueba,0,0
2,Validación ∩ prueba,0,0
3,Suma de tamaños,10000,10000


## 5. Escalado y codificación

- `StandardScaler`: `CreditScore`, `Age`, `Tenure`, `Balance`, `NumOfProducts` y `EstimatedSalary`.
- `OneHotEncoder(handle_unknown="ignore")`: `Geography`.
- Passthrough sin escalado: `HasCrCard` e `IsActiveMember`.

El preprocesador se ajusta una sola vez con `X_train`; validación y prueba reciben únicamente `transform`.

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), NUMERIC_FEATURES),
        (
            "geography",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            CATEGORICAL_FEATURES,
        ),
        ("binary", "passthrough", BINARY_FEATURES),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

X_train_processed = preprocessor.fit_transform(X_train).astype(np.float32)
X_val_processed = preprocessor.transform(X_val).astype(np.float32)
X_test_processed = preprocessor.transform(X_test).astype(np.float32)

y_train_array = y_train.to_numpy(dtype=np.float32)
y_val_array = y_val.to_numpy(dtype=np.float32)
y_test_array = y_test.to_numpy(dtype=np.float32)
feature_names = preprocessor.get_feature_names_out()

transformed_shapes = pd.DataFrame({
    "conjunto": ["Entrenamiento", "Validación", "Prueba"],
    "forma_X": [X_train_processed.shape, X_val_processed.shape, X_test_processed.shape],
    "forma_y": [y_train_array.shape, y_val_array.shape, y_test_array.shape],
    "tipo_X": [X_train_processed.dtype, X_val_processed.dtype, X_test_processed.dtype],
})
display(transformed_shapes)
print("Características transformadas:", feature_names.tolist())

,conjunto,forma_X,forma_y,tipo_X
0,Entrenamiento,"(7000, 11)","(7000,)",float32
1,Validación,"(1500, 11)","(1500,)",float32
2,Prueba,"(1500, 11)","(1500,)",float32


Características transformadas: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary', 'Geography_France', 'Geography_Germany', 'Geography_Spain', 'HasCrCard', 'IsActiveMember']


In [7]:
scaler = preprocessor.named_transformers_["numeric"]
encoder = preprocessor.named_transformers_["geography"]

has_card_index = feature_names.tolist().index("HasCrCard")
active_member_index = feature_names.tolist().index("IsActiveMember")

preprocessing_checks = pd.DataFrame({
    "comprobación": [
        "11 características en los tres conjuntos",
        "Matrices float32", "Matrices sin NaN o infinitos",
        "Scaler ajustado con 7.000 registros",
        "Medias corresponden solo a X_train",
        "HasCrCard conservada", "IsActiveMember conservada",
        "Tres categorías aprendidas para Geography",
    ],
    "resultado": [
        X_train_processed.shape[1] == X_val_processed.shape[1] == X_test_processed.shape[1] == 11,
        X_train_processed.dtype == X_val_processed.dtype == X_test_processed.dtype == np.float32,
        bool(
            np.isfinite(X_train_processed).all()
            and np.isfinite(X_val_processed).all()
            and np.isfinite(X_test_processed).all()
        ),
        int(scaler.n_samples_seen_) == len(X_train),
        bool(np.allclose(scaler.mean_, X_train[NUMERIC_FEATURES].mean().to_numpy())),
        bool(np.array_equal(X_train_processed[:, has_card_index], X_train["HasCrCard"].to_numpy(np.float32))),
        bool(np.array_equal(X_train_processed[:, active_member_index], X_train["IsActiveMember"].to_numpy(np.float32))),
        encoder.categories_[0].tolist() == ["France", "Germany", "Spain"],
    ],
})
display(preprocessing_checks)

scaler_parameters = pd.DataFrame({
    "variable": NUMERIC_FEATURES,
    "media_entrenamiento": scaler.mean_,
    "desviación_entrenamiento": scaler.scale_,
})
display(scaler_parameters)

,comprobación,resultado
0,11 características en los tres conjuntos,True
1,Matrices float32,True
2,Matrices sin NaN o infinitos,True
3,Scaler ajustado con 7.000 registros,True
4,Medias corresponden solo a X_train,True
5,HasCrCard conservada,True
6,IsActiveMember conservada,True
7,Tres categorías aprendidas para Geography,True


,variable,media_entrenamiento,desviación_entrenamiento
0,CreditScore,650.562429,96.532561
1,Age,38.998429,10.574194
2,Tenure,5.010571,2.891693
3,Balance,76314.462261,62370.118628
4,NumOfProducts,1.533000,0.584120
5,EstimatedSalary,99466.316050,57447.307427


## 6. Persistencia de matrices, preprocesador y metadatos

Las matrices se guardan con `numpy.savez_compressed` e incluyen `X`, `y`, los índices originales y un identificador común. El preprocesador ajustado se serializa con Joblib y se vuelve a cargar para verificar equivalencia. Solo deben cargarse archivos Joblib producidos por una fuente confiable, porque este formato puede ejecutar código durante la deserialización.

In [8]:
source_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
contract = {
    "source_sha256": source_sha256,
    "schema_version": SCHEMA_VERSION,
    "random_state": RANDOM_STATE,
    "features": FEATURES,
    "transformed_features": feature_names.tolist(),
}
artifact_id = hashlib.sha256(
    json.dumps(contract, sort_keys=True).encode("utf-8")
).hexdigest()[:16]

np.savez_compressed(
    TRAIN_PATH,
    X=X_train_processed,
    y=y_train_array,
    indices=X_train.index.to_numpy(dtype=np.int64),
    artifact_id=np.array(artifact_id),
)
np.savez_compressed(
    VALIDATION_PATH,
    X=X_val_processed,
    y=y_val_array,
    indices=X_val.index.to_numpy(dtype=np.int64),
    artifact_id=np.array(artifact_id),
)
np.savez_compressed(
    TEST_PATH,
    X=X_test_processed,
    y=y_test_array,
    indices=X_test.index.to_numpy(dtype=np.int64),
    artifact_id=np.array(artifact_id),
)

preprocessor.artifact_id_ = artifact_id
preprocessor.preprocessing_schema_version_ = SCHEMA_VERSION
preprocessor.source_sha256_ = source_sha256
joblib.dump(preprocessor, PREPROCESSOR_PATH)

metadata = {
    "preprocessing_schema_version": SCHEMA_VERSION,
    "artifact_id": artifact_id,
    "source": {
        "filename": DATA_FILENAME,
        "sha256": source_sha256,
        "records": len(df_original),
        "columns": df_original.shape[1],
    },
    "features": FEATURES,
    "transformed_features": feature_names.tolist(),
    "split": {
        "random_state": RANDOM_STATE,
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15,
    },
    "datasets": {
        "train": {"file": TRAIN_PATH.name, "X_shape": list(X_train_processed.shape), "y_shape": list(y_train_array.shape)},
        "validation": {"file": VALIDATION_PATH.name, "X_shape": list(X_val_processed.shape), "y_shape": list(y_val_array.shape)},
        "test": {"file": TEST_PATH.name, "X_shape": list(X_test_processed.shape), "y_shape": list(y_test_array.shape)},
    },
    "scikit_learn_version": sklearn.__version__,
    "preprocessor_file": PREPROCESSOR_PATH.name,
    "clean_dataset_file": CLEAN_DATA_PATH.name,
}
METADATA_PATH.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print("Identificador de artefactos:", artifact_id)
print("SHA-256 del CSV original:", source_sha256)
print("Metadatos:", METADATA_PATH.as_posix())

Identificador de artefactos: 3a2890fbab71e9c8
SHA-256 del CSV original: 92269a89d774dfd9065f9b8d04a05e7ea647da289c1a4a83a7893ea50a1adc3e
Metadatos: ../data/processed/metadata.json


## 7. Verificación de artefactos exportados

Se vuelven a abrir el CSV, los NPZ, los metadatos y el preprocesador confiable recién creado. La comprobación confirma dimensiones, tipos, trazabilidad, identidad común y equivalencia numérica con los objetos en memoria.

In [9]:
with np.load(TRAIN_PATH, allow_pickle=False) as train_artifact:
    saved_train = {key: train_artifact[key] for key in train_artifact.files}
with np.load(VALIDATION_PATH, allow_pickle=False) as validation_artifact:
    saved_validation = {key: validation_artifact[key] for key in validation_artifact.files}
with np.load(TEST_PATH, allow_pickle=False) as test_artifact:
    saved_test = {key: test_artifact[key] for key in test_artifact.files}

saved_metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
trusted_preprocessor = joblib.load(PREPROCESSOR_PATH)
reloaded_validation_sample = trusted_preprocessor.transform(X_val.iloc[:10]).astype(np.float32)

artifact_checks = pd.DataFrame({
    "comprobación": [
        "train.npz equivalente a memoria",
        "validation.npz equivalente a memoria",
        "test.npz equivalente a memoria",
        "Índices originales conservados",
        "Identificador común en NPZ y metadata",
        "Preprocesador ajustado y con 11 salidas",
        "Preprocesador recargado equivalente",
        "Preprocesador y metadata corresponden",
        "SHA-256 registrado corresponde al original",
    ],
    "resultado": [
        bool(np.array_equal(saved_train["X"], X_train_processed) and np.array_equal(saved_train["y"], y_train_array)),
        bool(np.array_equal(saved_validation["X"], X_val_processed) and np.array_equal(saved_validation["y"], y_val_array)),
        bool(np.array_equal(saved_test["X"], X_test_processed) and np.array_equal(saved_test["y"], y_test_array)),
        bool(
            np.array_equal(saved_train["indices"], X_train.index.to_numpy(np.int64))
            and np.array_equal(saved_validation["indices"], X_val.index.to_numpy(np.int64))
            and np.array_equal(saved_test["indices"], X_test.index.to_numpy(np.int64))
        ),
        bool(
            saved_train["artifact_id"].item()
            == saved_validation["artifact_id"].item()
            == saved_test["artifact_id"].item()
            == saved_metadata["artifact_id"]
        ),
        int(trusted_preprocessor.named_transformers_["numeric"].n_samples_seen_) == 7_000
        and len(trusted_preprocessor.get_feature_names_out()) == 11,
        bool(np.allclose(reloaded_validation_sample, X_val_processed[:10])),
        trusted_preprocessor.artifact_id_ == saved_metadata["artifact_id"],
        saved_metadata["source"]["sha256"] == source_sha256,
    ],
})
display(artifact_checks)

artifact_paths = [
    CLEAN_DATA_PATH, TRAIN_PATH, VALIDATION_PATH, TEST_PATH,
    METADATA_PATH, PREPROCESSOR_PATH,
]
artifact_sizes = pd.DataFrame({
    "archivo": [path.as_posix() for path in artifact_paths],
    "tamaño_KiB": [path.stat().st_size / 1024 for path in artifact_paths],
})
display(artifact_sizes.round(2))

,comprobación,resultado
0,train.npz equivalente a memoria,True
1,validation.npz equivalente a memoria,True
2,test.npz equivalente a memoria,True
3,Índices originales conservados,True
4,Identificador común en NPZ y metadata,True
5,Preprocesador ajustado y con 11 salidas,True
6,Preprocesador recargado equivalente,True
7,Preprocesador y metadata corresponden,True
8,SHA-256 registrado corresponde al original,True


,archivo,tamaño_KiB
0,../data/processed/clientes_limpios.csv,1134.00
1,../data/processed/train.npz,121.01
2,../data/processed/validation.npz,27.20
3,../data/processed/test.npz,27.33
4,../data/processed/metadata.json,1.42
5,../models/preprocesador.joblib,3.38


## 8. Resultado

El notebook produce conjuntamente:

- `data/processed/clientes_limpios.csv`: 10.000 × 21, sin escalado ni one-hot.
- `data/processed/train.npz`: entrenamiento transformado y trazable.
- `data/processed/validation.npz`: validación transformada y trazable.
- `data/processed/test.npz`: prueba transformada y reservada para evaluación posterior.
- `data/processed/metadata.json`: contrato de datos y versiones.
- `models/preprocesador.joblib`: `ColumnTransformer` ajustado exclusivamente con entrenamiento.

Estos archivos deben regenerarse juntos cuando cambie el dataset o el procedimiento. El siguiente paso reproducible es ejecutar `03_red_neuronal.ipynb`, que consume únicamente entrenamiento, validación y metadatos; no vuelve a limpiar ni ajustar transformadores.